In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure
import seaborn as sns

from tabulate import tabulate
from prettytable import PrettyTable, TableStyle
from IPython.display import display, HTML

from nhlpy.nhl_client import NHLClient

from nhlpy.api.query.builder import QueryBuilder, QueryContext
from nhlpy.api.query.filters.draft import DraftQuery
from nhlpy.api.query.filters.season import SeasonQuery
from nhlpy.api.query.filters.game_type import GameTypeQuery
from nhlpy.api.query.filters.position import PositionQuery, PositionTypes
from nhlpy.api.query.filters.franchise import FranchiseQuery
from nhlpy.api.query.filters.shoot_catch import ShootCatchesQuery
from nhlpy.api.query.filters.status import StatusQuery
from nhlpy.api.query.filters.opponent import OpponentQuery
from nhlpy.api.query.filters.home_road import HomeRoadQuery
from nhlpy.api.query.filters.experience import ExperienceQuery
from nhlpy.api.query.filters.decision import DecisionQuery

# custom functions from this repo
from utils.function_library import find_player_id
from utils.function_library import toi_string_to_float
from utils.function_library import load_summary_statistics_for_skaters
from utils.function_library import load_realtime_statistics_for_skaters
from utils.function_library import load_faceoffwins_statistics_for_skaters
from utils.function_library import skater_single_season_fantasy_points
from utils.function_library import get_stats_by_season
from utils.function_library import plot_stat_per_game
from utils.function_library import home_away_split

from matplotlib import rc
plt.rcParams.update({'font.size':22})

RED_BOLD = "\033[1;31m"
GREEN_BOLD = "\033[1;92m"
RESET = "\033[0m"


In [2]:
from decimal import Decimal

def format_table_stats(stat_arr, stat_name):

    idx_max = np.argmax(stat_arr)+1
    idx_min = np.argmin(stat_arr)+1

    stat_arr_obj = stat_arr.astype(object)
    #stat_arr_obj = np.array([f"{x:.2f}" for x in stat_arr_obj], dtype=object)
    stat_arr_obj = np.array([f"{x:g}" for x in stat_arr_obj], dtype=object)
    stat_arr_obj = np.insert(stat_arr_obj, 0, stat_name)
    '''
    # Find index of the largest value
    stat_arr = np.insert(stat_arr, 0, -9999.)
    idx_max = np.argmax(stat_arr)
    # Find index of the smallest value    
    stat_arr[0] = np.inf
    idx_min = np.argmin(stat_arr)
    '''
    # Create row data, bolding the largest value
    '''
    row_data = [
        str(val) if i != idx_max else f"\033[1;92m{val}\033[0m"
        #str(val) if i != idx_max f"\033[1;31m{val}\033[0m" elif i != idx_min else f"\033[1;92m{val}\033[0m"
        for i, val in enumerate(stat_arr_obj)
    ]
    '''
    row_data = [""] * len(stat_arr_obj)
    for i, val in enumerate(stat_arr_obj):
        if i == idx_max:
            row_data[i] = f"\033[1;92m{val}\033[0m"
        elif i == idx_min:
            row_data[i] = f"\033[1;31m{val}\033[0m"
        else:
            row_data[i] = str(val)
        
    return row_data


In [3]:
fp_goals = 6.0
fp_assists = 4.0
fp_plusminus = 1.5
fp_pp_goals = 2.0
fp_pp_assists = 2.0
fp_sh_goals = 4.0
fp_sh_assists = 1.0
fp_game_winning_goals = 1.0
fp_shots = 0.75
fp_hits = 0.40
fp_blocks = 1.25
fp_fowins = 0.15
fp_folosses = -0.15
fp_pims = 0.0

season = "20252026"


In [4]:
skater_names = ["Collin Graf", "Tyler Toffoli", "Will Smith", "Michael Misa"]
'''
skater_ids = [""] * len(skater_names)

for i in range(len(skater_names)):
    skater_ids[i] = find_player_id(skater_names[i], season) # do i need to wrap find_player_id in str()
'''

'\nskater_ids = [""] * len(skater_names)\n\nfor i in range(len(skater_names)):\n    skater_ids[i] = find_player_id(skater_names[i], season) # do i need to wrap find_player_id in str()\n'

In [ ]:
#skater_ids

In [ ]:
skater_summary_query = load_summary_statistics_for_skaters(season, season)
skater_realtime_query = load_realtime_statistics_for_skaters(season, season)
skater_faceoffwins_query = load_faceoffwins_statistics_for_skaters(season, season)

In [ ]:
skater_names_grabbed = ["N/A"] * len(skater_names)
skater_season_gamesPlayed = np.zeros(len(skater_names))
skater_season_timeOnIcePerGame = np.zeros(len(skater_names))
skater_season_timeOnIcePerGame_min = np.zeros(len(skater_names))
skater_season_goals = np.zeros(len(skater_names))
skater_season_assists = np.zeros(len(skater_names))
skater_season_plusMinus = np.zeros(len(skater_names))
skater_season_ppGoals = np.zeros(len(skater_names))
skater_season_ppAssists = np.zeros(len(skater_names))
skater_season_shGoals = np.zeros(len(skater_names))
skater_season_shAssists = np.zeros(len(skater_names))
skater_season_gameWinningGoals = np.zeros(len(skater_names))
skater_season_shots = np.zeros(len(skater_names))
skater_season_penaltyMinutes = np.zeros(len(skater_names))
skater_season_blockedShots = np.zeros(len(skater_names))
skater_season_hits = np.zeros(len(skater_names))
skater_season_totalFaceoffWins = np.zeros(len(skater_names))
skater_season_totalFaceoffLosses = np.zeros(len(skater_names))

for name in range(len(skater_names)):
    for i in range(len(skater_summary_query)):
        if skater_summary_query[i]["skaterFullName"] == skater_names[name]:
            skater_names_grabbed[name] = skater_summary_query[i]["skaterFullName"]
            skater_season_timeOnIcePerGame[name] = skater_summary_query[i]["timeOnIcePerGame"]
            skater_season_timeOnIcePerGame_min[name] = round(skater_summary_query[i]["timeOnIcePerGame"]/60., 2)
            skater_season_gamesPlayed[name] = skater_summary_query[i]["gamesPlayed"]
            skater_season_goals[name] = skater_summary_query[i]["goals"]
            skater_season_assists[name] = skater_summary_query[i]["assists"]
            skater_season_plusMinus[name] = skater_summary_query[i]["plusMinus"]
            skater_season_ppGoals[name] = skater_summary_query[i]["ppGoals"]
            skater_season_ppAssists[name] = skater_summary_query[i]["ppPoints"] - skater_summary_query[i]["ppGoals"]
            skater_season_shGoals[name] = skater_summary_query[i]["shGoals"]
            skater_season_shAssists[name] = skater_summary_query[i]["shPoints"] - skater_summary_query[i]["shGoals"]
            skater_season_gameWinningGoals[name] = skater_summary_query[i]["gameWinningGoals"]
            skater_season_shots[name] = skater_summary_query[i]["shots"]
            skater_season_penaltyMinutes[name] = skater_summary_query[i]["penaltyMinutes"]

    for i in range(len(skater_realtime_query)):
        if skater_realtime_query[i]["skaterFullName"] == skater_names[name]:
            skater_season_blockedShots[name] = skater_realtime_query[i]["blockedShots"]
            skater_season_hits[name] = skater_realtime_query[i]["hits"]

    for i in range(len(skater_faceoffwins_query)):
        if skater_faceoffwins_query[i]["skaterFullName"] == skater_names[name]:
            skater_season_totalFaceoffWins[name] = skater_faceoffwins_query[i]["totalFaceoffWins"]
            skater_season_totalFaceoffLosses[name] = skater_faceoffwins_query[i]["totalFaceoffLosses"]


In [ ]:
"""
# get the stat / game
skater_season_goals_per_game = skater_season_goals/skater_season_gamesPlayed
skater_season_assists_per_game = skater_season_assists/skater_season_gamesPlayed
skater_season_plusMinus_per_game = skater_season_plusMinus/skater_season_gamesPlayed
skater_season_ppGoals_per_game = skater_season_ppGoals/skater_season_gamesPlayed
skater_season_ppAssists_per_game = skater_season_ppAssists/skater_season_gamesPlayed
skater_season_shGoals_per_game = skater_season_shGoals/skater_season_gamesPlayed
skater_season_shAssists_per_game = skater_season_shAssists/skater_season_gamesPlayed
skater_season_gameWinningGoals_per_game = skater_season_gameWinningGoals/skater_season_gamesPlayed
skater_season_shots_per_game = skater_season_shots/skater_season_gamesPlayed
skater_season_penaltyMinutes_per_game = skater_season_penaltyMinutes/skater_season_gamesPlayed
skater_season_blockedShots_per_game = skater_season_blockedShots/skater_season_gamesPlayed
skater_season_hits_per_game = skater_season_hits/skater_season_gamesPlayed
skater_season_totalFaceoffWins_per_game = skater_season_totalFaceoffWins/skater_season_gamesPlayed
skater_season_totalFaceoffLosses_per_game = skater_season_totalFaceoffLosses/skater_season_gamesPlayed
"""
# get the stat / game
skater_season_goals_per_game = np.zeros(len(skater_names))
skater_season_assists_per_game = np.zeros(len(skater_names))
skater_season_plusMinus_per_game = np.zeros(len(skater_names))
skater_season_ppGoals_per_game = np.zeros(len(skater_names))
skater_season_ppAssists_per_game = np.zeros(len(skater_names))
skater_season_shGoals_per_game = np.zeros(len(skater_names))
skater_season_shAssists_per_game = np.zeros(len(skater_names))
skater_season_gameWinningGoals_per_game = np.zeros(len(skater_names))
skater_season_shots_per_game = np.zeros(len(skater_names))
skater_season_penaltyMinutes_per_game = np.zeros(len(skater_names))
skater_season_blockedShots_per_game = np.zeros(len(skater_names))
skater_season_hits_per_game = np.zeros(len(skater_names))
skater_season_totalFaceoffWins_per_game = np.zeros(len(skater_names))
skater_season_totalFaceoffLosses_per_game = np.zeros(len(skater_names))
skater_season_penaltyMinutes_per_game = np.zeros(len(skater_names))

# get the stat / 60 minutes
skater_season_goals_per_60min = np.zeros(len(skater_names))
skater_season_assists_per_60min = np.zeros(len(skater_names))
skater_season_plusMinus_per_60min = np.zeros(len(skater_names))
skater_season_ppGoals_per_60min = np.zeros(len(skater_names))
skater_season_ppAssists_per_60min = np.zeros(len(skater_names))
skater_season_shGoals_per_60min = np.zeros(len(skater_names))
skater_season_shAssists_per_60min = np.zeros(len(skater_names))
skater_season_gameWinningGoals_per_60min = np.zeros(len(skater_names))
skater_season_shots_per_60min = np.zeros(len(skater_names))
skater_season_penaltyMinutes_per_60min = np.zeros(len(skater_names))
skater_season_blockedShots_per_60min = np.zeros(len(skater_names))
skater_season_hits_per_60min = np.zeros(len(skater_names))
skater_season_totalFaceoffWins_per_60min = np.zeros(len(skater_names))
skater_season_totalFaceoffLosses_per_60min = np.zeros(len(skater_names))
skater_season_penaltyMinutes_per_60min = np.zeros(len(skater_names))


In [ ]:
skater_names_tot_fantasy_points = np.zeros(len(skater_names))
skater_names_fantasy_points_per_game = np.zeros(len(skater_names))

for name in range(len(skater_names)):
    if skater_names[name] == "N/A":
        print("Skater %s not found: error in defining player name or season" % (skater_names[name]))
    else:
        skater_names_tot_fantasy_points[name] = (skater_season_goals[name]*fp_goals
            + skater_season_assists[name]*fp_assists
            + skater_season_plusMinus[name]*fp_plusminus
            + skater_season_ppGoals[name]*fp_pp_goals
            + skater_season_ppAssists[name]*fp_pp_assists
            + skater_season_shGoals[name]*fp_sh_goals
            + skater_season_shAssists[name]*fp_sh_assists
            + skater_season_gameWinningGoals[name]*fp_game_winning_goals
            + skater_season_shots[name]*fp_shots
            + skater_season_hits[name]*fp_hits
            + skater_season_blockedShots[name]*fp_blocks
            + skater_season_totalFaceoffWins[name]*fp_fowins
            + skater_season_totalFaceoffLosses[name]*fp_folosses
            + skater_season_penaltyMinutes[name]*fp_pims
            )
        skater_names_fantasy_points_per_game[name] = round(skater_names_tot_fantasy_points[name]/skater_season_gamesPlayed[name], 2)

        # get stats per game
        skater_season_goals_per_game[name] = round(skater_season_goals[name]/skater_season_gamesPlayed[name], 2)
        skater_season_assists_per_game[name] = round(skater_season_assists[name]/skater_season_gamesPlayed[name], 2)
        skater_season_plusMinus_per_game[name] = round(skater_season_plusMinus[name]/skater_season_gamesPlayed[name], 2)
        skater_season_ppGoals_per_game[name] = round(skater_season_ppGoals[name]/skater_season_gamesPlayed[name], 2)
        skater_season_ppAssists_per_game[name] = round(skater_season_ppAssists[name]/skater_season_gamesPlayed[name], 2)
        skater_season_shGoals_per_game[name] = round(skater_season_shGoals[name]/skater_season_gamesPlayed[name], 2)
        skater_season_shAssists_per_game[name] = round(skater_season_shAssists[name]/skater_season_gamesPlayed[name], 2)
        skater_season_gameWinningGoals_per_game[name] = round(skater_season_gameWinningGoals[name]/skater_season_gamesPlayed[name], 2)
        skater_season_shots_per_game[name] = round(skater_season_shots[name]/skater_season_gamesPlayed[name], 2)
        skater_season_hits_per_game[name] = round(skater_season_hits[name]/skater_season_gamesPlayed[name], 2)
        skater_season_blockedShots_per_game[name] = round(skater_season_blockedShots[name]/skater_season_gamesPlayed[name], 2)
        skater_season_totalFaceoffWins_per_game[name] = round(skater_season_totalFaceoffWins[name]/skater_season_gamesPlayed[name], 2)
        skater_season_totalFaceoffLosses_per_game[name] = round(skater_season_totalFaceoffLosses[name]/skater_season_gamesPlayed[name], 2)
        skater_season_penaltyMinutes_per_game[name] = round(skater_season_penaltyMinutes[name]/skater_season_gamesPlayed[name], 2)

        # get stats per 60 minutes 
        skater_season_goals_per_60min[name] = round(60.*skater_season_goals_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
        skater_season_assists_per_60min[name] = round(60.*skater_season_assists_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
        skater_season_plusMinus_per_60min[name] = round(60.*skater_season_plusMinus_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
        skater_season_ppGoals_per_60min[name] = round(60.*skater_season_ppGoals_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
        skater_season_ppAssists_per_60min[name] = round(60.*skater_season_ppAssists_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
        skater_season_shGoals_per_60min[name] = round(60.*skater_season_shGoals_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
        skater_season_shAssists_per_60min[name] = round(60.*skater_season_shAssists_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
        skater_season_gameWinningGoals_per_60min[name] = round(60.*skater_season_gameWinningGoals_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
        skater_season_shots_per_60min[name] = round(60.*skater_season_shots_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
        skater_season_hits_per_60min[name] = round(60.*skater_season_hits_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
        skater_season_blockedShots_per_60min[name] = round(60.*skater_season_blockedShots_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
        skater_season_totalFaceoffWins_per_60min[name] = round(60.*skater_season_totalFaceoffWins_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
        skater_season_totalFaceoffLosses_per_60min[name] = round(60.*skater_season_totalFaceoffLosses_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
        skater_season_penaltyMinutes_per_60min[name] = round(60.*skater_season_penaltyMinutes_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)


In [ ]:
skater_names_grabbed.insert(0, "Stat")

# fantasy point table

In [ ]:
fp_table_headers = [
    f"\033[1m{skater_names_grabbed[i]}\033[0m" for i in range(len(skater_names_grabbed))
]

# Initialize PrettyTable with the dynamic headers
fp_table = PrettyTable(fp_table_headers)
fp_table.title = "\033[1mPlayer Fantasy Comparison\033[0m"

table_fantasy_points_total = format_table_stats(skater_names_tot_fantasy_points, "Total Fantasy Points")
fp_table.add_row(table_fantasy_points_total)

table_fantasy_points_total = format_table_stats(skater_names_fantasy_points_per_game, "# of Fantasy Points per Game")
fp_table.add_row(table_fantasy_points_total)

print(fp_table)


# real stats table

In [ ]:
stats_headers = [
    f"\033[1m{skater_names_grabbed[i]}\033[0m" for i in range(len(skater_names_grabbed))
]

# Initialize PrettyTable with the dynamic headers
stats_table = PrettyTable(stats_headers)
stats_table.title = "\033[1mPlayer Stats Comparison\033[0m"

# games played
table_season_games_played_total = format_table_stats(skater_season_gamesPlayed, "Games Played")
stats_table.add_row(table_season_games_played_total)
# average TOI
table_season_toi_per_game = format_table_stats(skater_season_timeOnIcePerGame_min, "Average TOI")
stats_table.add_row(table_season_toi_per_game)
# goals
stats_table.add_divider()
table_season_goals_total = format_table_stats(skater_season_goals, "Total Goals")
stats_table.add_row(table_season_goals_total)
table_season_goals_per_game = format_table_stats(skater_season_goals_per_game, "# of Goals per Game")
stats_table.add_row(table_season_goals_per_game)
table_season_goals_per_60min = format_table_stats(skater_season_goals_per_60min, "# of Goals per 60 minutes")
stats_table.add_row(table_season_goals_per_60min)
# assists
stats_table.add_divider()
table_season_assists_total = format_table_stats(skater_season_assists, "Total Assists")
stats_table.add_row(table_season_assists_total)
table_season_assists_per_game = format_table_stats(skater_season_assists_per_game, "# of Assists per Game")
stats_table.add_row(table_season_assists_per_game)
table_season_assists_per_60min = format_table_stats(skater_season_assists_per_60min, "# of Assists per 60 minutes")
stats_table.add_row(table_season_assists_per_60min)
# +/-
stats_table.add_divider()
table_season_plusMinus_total = format_table_stats(skater_season_plusMinus, "Season +/-")
stats_table.add_row(table_season_plusMinus_total)
table_season_plusMinus_per_game = format_table_stats(skater_season_plusMinus_per_game, "Average +/- per Game")
stats_table.add_row(table_season_plusMinus_per_game)
table_season_plusMinus_per_60min = format_table_stats(skater_season_plusMinus_per_60min, "Average +/- per 60 minutes")
stats_table.add_row(table_season_plusMinus_per_60min)
# PP Goals
stats_table.add_divider()
table_season_ppGoals_total = format_table_stats(skater_season_ppGoals, "Season PP Goals")
stats_table.add_row(table_season_ppGoals_total)
table_season_ppGoals_per_game = format_table_stats(skater_season_ppGoals_per_game, "Average PP Goals per Game")
stats_table.add_row(table_season_ppGoals_per_game)
table_season_ppGoals_per_60min = format_table_stats(skater_season_ppGoals_per_60min, "Average # of PP Goals per 60 minutes")
stats_table.add_row(table_season_ppGoals_per_60min)
# PP Assists
stats_table.add_divider()
table_season_ppAssists_total = format_table_stats(skater_season_ppAssists, "Season PP Assists")
stats_table.add_row(table_season_ppAssists_total)
table_season_ppAssists_per_game = format_table_stats(skater_season_ppAssists_per_game, "Average PP Assists per Game")
stats_table.add_row(table_season_ppAssists_per_game)
table_season_ppAssists_per_60min = format_table_stats(skater_season_ppAssists_per_60min, "Average # of PP Assists per 60 minutes")
stats_table.add_row(table_season_ppAssists_per_60min)
# SH Goals
stats_table.add_divider()
table_season_shGoals_total = format_table_stats(skater_season_shGoals, "Season SH Goals")
stats_table.add_row(table_season_shGoals_total)
table_season_shGoals_per_game = format_table_stats(skater_season_shGoals_per_game, "Average SH Goals per Game")
stats_table.add_row(table_season_shGoals_per_game)
table_season_shGoals_per_60min = format_table_stats(skater_season_shGoals_per_60min, "# of SH Goals per 60 minutes")
stats_table.add_row(table_season_shGoals_per_60min)
# SH Assists
stats_table.add_divider()
table_season_shAssists_total = format_table_stats(skater_season_shAssists, "Season SH Assists")
stats_table.add_row(table_season_shAssists_total)
table_season_shAssists_per_game = format_table_stats(skater_season_shAssists_per_game, "Average SH Assists per Game")
stats_table.add_row(table_season_shAssists_per_game)
table_season_shAssists_per_60min = format_table_stats(skater_season_shAssists_per_60min, "Average # of SH Assists per 60 minutes")
stats_table.add_row(table_season_shAssists_per_60min)
# Game Winning Goals
stats_table.add_divider()
table_season_gameWinningGoals_total = format_table_stats(skater_season_gameWinningGoals, "Season Game Winning Goals")
stats_table.add_row(table_season_gameWinningGoals_total)
table_season_gameWinningGoals_per_game = format_table_stats(skater_season_gameWinningGoals_per_game, "Average # of Game Winning Goals per Game")
stats_table.add_row(table_season_gameWinningGoals_per_game)
table_season_gameWinningGoals_per_60min = format_table_stats(skater_season_gameWinningGoals_per_60min, "Average # of Game Winning Goals per 60 minutes")
stats_table.add_row(table_season_gameWinningGoals_per_60min)
# Shots
stats_table.add_divider()
table_season_shots_total = format_table_stats(skater_season_shots, "Season Shots")
stats_table.add_row(table_season_shots_total)
table_season_shots_per_game = format_table_stats(skater_season_shots_per_game, "Average # of Shots per Game")
stats_table.add_row(table_season_shots_per_game)
table_season_shots_per_60min = format_table_stats(skater_season_shots_per_60min, "Average # of Shots per 60 minutes")
stats_table.add_row(table_season_shots_per_60min)
# Hits
stats_table.add_divider()
table_season_hits_total = format_table_stats(skater_season_hits, "Season Hits")
stats_table.add_row(table_season_hits_total)
table_season_hits_per_game = format_table_stats(skater_season_hits_per_game, "Average # of Hits per Game")
stats_table.add_row(table_season_hits_per_game)
table_season_hits_per_60min = format_table_stats(skater_season_hits_per_60min, "Average # of Hits per 60 minutes")
stats_table.add_row(table_season_hits_per_60min)
# Blocked Shots
stats_table.add_divider()
table_season_blockedShots_total = format_table_stats(skater_season_blockedShots, "Season Blocked Shots")
stats_table.add_row(table_season_blockedShots_total)
table_season_blockedShots_per_game = format_table_stats(skater_season_blockedShots_per_game, "Average # of Blocked Shots per Game")
stats_table.add_row(table_season_blockedShots_per_game)
table_season_blockedShots_per_60min = format_table_stats(skater_season_blockedShots_per_60min, "Average # of Blocked Shots per 60 minutes")
stats_table.add_row(table_season_blockedShots_per_60min)
# FO Wins
stats_table.add_divider()
table_season_totalFaceoffWins_total = format_table_stats(skater_season_totalFaceoffWins, "Season FO Wins")
stats_table.add_row(table_season_totalFaceoffWins_total)
table_season_totalFaceoffWins_per_game = format_table_stats(skater_season_totalFaceoffWins_per_game, "Average # of FO Wins per Game")
stats_table.add_row(table_season_totalFaceoffWins_per_game)
table_season_totalFaceoffWins_per_60min = format_table_stats(skater_season_totalFaceoffWins_per_60min, "Average # of FO Wins per 60 minutes")
stats_table.add_row(table_season_totalFaceoffWins_per_60min)
# FO Loses
stats_table.add_divider()
table_season_totalFaceoffLosses_total = format_table_stats(skater_season_totalFaceoffLosses, "Season FO Losses")
stats_table.add_row(table_season_totalFaceoffLosses_total)
table_season_totalFaceoffLosses_per_game = format_table_stats(skater_season_totalFaceoffLosses_per_game, "Average # of FO Losses per Game")
stats_table.add_row(table_season_totalFaceoffLosses_per_game)
table_season_totalFaceoffLosses_per_60min = format_table_stats(skater_season_totalFaceoffLosses_per_60min, "Average # of FO Losses per 60 minutes")
stats_table.add_row(table_season_totalFaceoffLosses_per_60min)
# PIMs
stats_table.add_divider()
table_season_penaltyMinutes_total = format_table_stats(skater_season_penaltyMinutes, "Season PIMs")
stats_table.add_row(table_season_penaltyMinutes_total)
table_season_penaltyMinutes_per_game = format_table_stats(skater_season_penaltyMinutes_per_game, "Average # of PIMs per Game")
stats_table.add_row(table_season_penaltyMinutes_per_game)
table_season_penaltyMinutes_per_60min = format_table_stats(skater_season_penaltyMinutes_per_60min, "Average # of PIMs per 60 minutes")
stats_table.add_row(table_season_penaltyMinutes_per_60min)


print(stats_table)

In [5]:
def comp_multi_skaters(skater_names,
    season,
    fantasy_points_goals,
    fantasy_points_assists,
    fantasy_points_plusminus,
    fantasy_points_pp_goals,
    fantasy_points_pp_asists,
    fantasy_points_sh_goals,
    fantasy_points_sh_assists,
    fantasy_points_game_winning_goals,
    fantasy_points_shots,
    fantasy_points_hits,
    fantasy_points_blocks,
    fantasy_points_fowins,
    fantasy_points_folosses,
    fantasy_points_pims,
    ):

    start_season = season
    end_season = season

    fantasy_goals = float(fantasy_points_goals)
    fantasy_assists = float(fantasy_points_assists)
    fantasy_plusminus = float(fantasy_points_plusminus)
    fantasy_pp_goals = float(fantasy_points_pp_goals)
    fantasy_pp_assists = float(fantasy_points_pp_asists)
    fantasy_sh_goals = float(fantasy_points_sh_goals)
    fantasy_sh_assists = float(fantasy_points_sh_assists)
    fantasy_game_winning_goals = float(fantasy_points_game_winning_goals)
    fantasy_shots = float(fantasy_points_shots)
    fantasy_hits = float(fantasy_points_hits)
    fantasy_blocks = float(fantasy_points_blocks)
    fantasy_fowins = float(fantasy_points_fowins)
    fantasy_folosses = float(fantasy_points_folosses)
    fantasy_pims = float(fantasy_points_pims)

    # do an initial query to see if player name exists

    skater_summary_query = load_summary_statistics_for_skaters(start_season, end_season)
    skater_realtime_query = load_realtime_statistics_for_skaters(start_season, end_season)
    skater_faceoffwins_query = load_faceoffwins_statistics_for_skaters(start_season, end_season)

    skater_names_grabbed = ["N/A"] * len(skater_names)
    skater_season_gamesPlayed = np.zeros(len(skater_names))
    skater_season_timeOnIcePerGame = np.zeros(len(skater_names))
    skater_season_timeOnIcePerGame_min = np.zeros(len(skater_names))
    skater_season_goals = np.zeros(len(skater_names))
    skater_season_assists = np.zeros(len(skater_names))
    skater_season_plusMinus = np.zeros(len(skater_names))
    skater_season_ppGoals = np.zeros(len(skater_names))
    skater_season_ppAssists = np.zeros(len(skater_names))
    skater_season_shGoals = np.zeros(len(skater_names))
    skater_season_shAssists = np.zeros(len(skater_names))
    skater_season_gameWinningGoals = np.zeros(len(skater_names))
    skater_season_shots = np.zeros(len(skater_names))
    skater_season_penaltyMinutes = np.zeros(len(skater_names))
    skater_season_blockedShots = np.zeros(len(skater_names))
    skater_season_hits = np.zeros(len(skater_names))
    skater_season_totalFaceoffWins = np.zeros(len(skater_names))
    skater_season_totalFaceoffLosses = np.zeros(len(skater_names))

    for name in range(len(skater_names)):
        for i in range(len(skater_summary_query)):
            if skater_summary_query[i]["skaterFullName"] == skater_names[name]:
                skater_names_grabbed[name] = skater_summary_query[i]["skaterFullName"]
                skater_season_timeOnIcePerGame[name] = skater_summary_query[i]["timeOnIcePerGame"]
                skater_season_timeOnIcePerGame_min[name] = round(skater_summary_query[i]["timeOnIcePerGame"]/60., 2)
                skater_season_gamesPlayed[name] = skater_summary_query[i]["gamesPlayed"]
                skater_season_goals[name] = skater_summary_query[i]["goals"]
                skater_season_assists[name] = skater_summary_query[i]["assists"]
                skater_season_plusMinus[name] = skater_summary_query[i]["plusMinus"]
                skater_season_ppGoals[name] = skater_summary_query[i]["ppGoals"]
                skater_season_ppAssists[name] = skater_summary_query[i]["ppPoints"] - skater_summary_query[i]["ppGoals"]
                skater_season_shGoals[name] = skater_summary_query[i]["shGoals"]
                skater_season_shAssists[name] = skater_summary_query[i]["shPoints"] - skater_summary_query[i]["shGoals"]
                skater_season_gameWinningGoals[name] = skater_summary_query[i]["gameWinningGoals"]
                skater_season_shots[name] = skater_summary_query[i]["shots"]
                skater_season_penaltyMinutes[name] = skater_summary_query[i]["penaltyMinutes"]

        for i in range(len(skater_realtime_query)):
            if skater_realtime_query[i]["skaterFullName"] == skater_names[name]:
                skater_season_blockedShots[name] = skater_realtime_query[i]["blockedShots"]
                skater_season_hits[name] = skater_realtime_query[i]["hits"]

        for i in range(len(skater_faceoffwins_query)):
            if skater_faceoffwins_query[i]["skaterFullName"] == skater_names[name]:
                skater_season_totalFaceoffWins[name] = skater_faceoffwins_query[i]["totalFaceoffWins"]
                skater_season_totalFaceoffLosses[name] = skater_faceoffwins_query[i]["totalFaceoffLosses"]

        # get the stat / game
        skater_season_goals_per_game = np.zeros(len(skater_names))
        skater_season_assists_per_game = np.zeros(len(skater_names))
        skater_season_plusMinus_per_game = np.zeros(len(skater_names))
        skater_season_ppGoals_per_game = np.zeros(len(skater_names))
        skater_season_ppAssists_per_game = np.zeros(len(skater_names))
        skater_season_shGoals_per_game = np.zeros(len(skater_names))
        skater_season_shAssists_per_game = np.zeros(len(skater_names))
        skater_season_gameWinningGoals_per_game = np.zeros(len(skater_names))
        skater_season_shots_per_game = np.zeros(len(skater_names))
        skater_season_penaltyMinutes_per_game = np.zeros(len(skater_names))
        skater_season_blockedShots_per_game = np.zeros(len(skater_names))
        skater_season_hits_per_game = np.zeros(len(skater_names))
        skater_season_totalFaceoffWins_per_game = np.zeros(len(skater_names))
        skater_season_totalFaceoffLosses_per_game = np.zeros(len(skater_names))
        skater_season_penaltyMinutes_per_game = np.zeros(len(skater_names))

        # get the stat / 60 minutes
        skater_season_goals_per_60min = np.zeros(len(skater_names))
        skater_season_assists_per_60min = np.zeros(len(skater_names))
        skater_season_plusMinus_per_60min = np.zeros(len(skater_names))
        skater_season_ppGoals_per_60min = np.zeros(len(skater_names))
        skater_season_ppAssists_per_60min = np.zeros(len(skater_names))
        skater_season_shGoals_per_60min = np.zeros(len(skater_names))
        skater_season_shAssists_per_60min = np.zeros(len(skater_names))
        skater_season_gameWinningGoals_per_60min = np.zeros(len(skater_names))
        skater_season_shots_per_60min = np.zeros(len(skater_names))
        skater_season_penaltyMinutes_per_60min = np.zeros(len(skater_names))
        skater_season_blockedShots_per_60min = np.zeros(len(skater_names))
        skater_season_hits_per_60min = np.zeros(len(skater_names))
        skater_season_totalFaceoffWins_per_60min = np.zeros(len(skater_names))
        skater_season_totalFaceoffLosses_per_60min = np.zeros(len(skater_names))
        skater_season_penaltyMinutes_per_60min = np.zeros(len(skater_names))

        skater_names_tot_fantasy_points = np.zeros(len(skater_names))
        skater_names_fantasy_points_per_game = np.zeros(len(skater_names))

    for name in range(len(skater_names)):
        if skater_names[name] == "N/A":
            print("Skater %s not found: error in defining player name or season" % (skater_names[name]))
        else:
            skater_names_tot_fantasy_points[name] = (skater_season_goals[name]*fp_goals
                + skater_season_assists[name]*fp_assists
                + skater_season_plusMinus[name]*fp_plusminus
                + skater_season_ppGoals[name]*fp_pp_goals
                + skater_season_ppAssists[name]*fp_pp_assists
                + skater_season_shGoals[name]*fp_sh_goals
                + skater_season_shAssists[name]*fp_sh_assists
                + skater_season_gameWinningGoals[name]*fp_game_winning_goals
                + skater_season_shots[name]*fp_shots
                + skater_season_hits[name]*fp_hits
                + skater_season_blockedShots[name]*fp_blocks
                + skater_season_totalFaceoffWins[name]*fp_fowins
                + skater_season_totalFaceoffLosses[name]*fp_folosses
                + skater_season_penaltyMinutes[name]*fp_pims
                )
            skater_names_fantasy_points_per_game[name] = round(skater_names_tot_fantasy_points[name]/skater_season_gamesPlayed[name], 2)

            # get stats per game
            skater_season_goals_per_game[name] = round(skater_season_goals[name]/skater_season_gamesPlayed[name], 2)
            skater_season_assists_per_game[name] = round(skater_season_assists[name]/skater_season_gamesPlayed[name], 2)
            skater_season_plusMinus_per_game[name] = round(skater_season_plusMinus[name]/skater_season_gamesPlayed[name], 2)
            skater_season_ppGoals_per_game[name] = round(skater_season_ppGoals[name]/skater_season_gamesPlayed[name], 2)
            skater_season_ppAssists_per_game[name] = round(skater_season_ppAssists[name]/skater_season_gamesPlayed[name], 2)
            skater_season_shGoals_per_game[name] = round(skater_season_shGoals[name]/skater_season_gamesPlayed[name], 2)
            skater_season_shAssists_per_game[name] = round(skater_season_shAssists[name]/skater_season_gamesPlayed[name], 2)
            skater_season_gameWinningGoals_per_game[name] = round(skater_season_gameWinningGoals[name]/skater_season_gamesPlayed[name], 2)
            skater_season_shots_per_game[name] = round(skater_season_shots[name]/skater_season_gamesPlayed[name], 2)
            skater_season_hits_per_game[name] = round(skater_season_hits[name]/skater_season_gamesPlayed[name], 2)
            skater_season_blockedShots_per_game[name] = round(skater_season_blockedShots[name]/skater_season_gamesPlayed[name], 2)
            skater_season_totalFaceoffWins_per_game[name] = round(skater_season_totalFaceoffWins[name]/skater_season_gamesPlayed[name], 2)
            skater_season_totalFaceoffLosses_per_game[name] = round(skater_season_totalFaceoffLosses[name]/skater_season_gamesPlayed[name], 2)
            skater_season_penaltyMinutes_per_game[name] = round(skater_season_penaltyMinutes[name]/skater_season_gamesPlayed[name], 2)

            # get stats per 60 minutes 
            skater_season_goals_per_60min[name] = round(60.*skater_season_goals_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
            skater_season_assists_per_60min[name] = round(60.*skater_season_assists_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
            skater_season_plusMinus_per_60min[name] = round(60.*skater_season_plusMinus_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
            skater_season_ppGoals_per_60min[name] = round(60.*skater_season_ppGoals_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
            skater_season_ppAssists_per_60min[name] = round(60.*skater_season_ppAssists_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
            skater_season_shGoals_per_60min[name] = round(60.*skater_season_shGoals_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
            skater_season_shAssists_per_60min[name] = round(60.*skater_season_shAssists_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
            skater_season_gameWinningGoals_per_60min[name] = round(60.*skater_season_gameWinningGoals_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
            skater_season_shots_per_60min[name] = round(60.*skater_season_shots_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
            skater_season_hits_per_60min[name] = round(60.*skater_season_hits_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
            skater_season_blockedShots_per_60min[name] = round(60.*skater_season_blockedShots_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
            skater_season_totalFaceoffWins_per_60min[name] = round(60.*skater_season_totalFaceoffWins_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
            skater_season_totalFaceoffLosses_per_60min[name] = round(60.*skater_season_totalFaceoffLosses_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)
            skater_season_penaltyMinutes_per_60min[name] = round(60.*skater_season_penaltyMinutes_per_game[name]/skater_season_timeOnIcePerGame_min[name], 2)

    skater_names_grabbed.insert(0, "Stat")

    fp_table_headers = [
        f"\033[1m{skater_names_grabbed[i]}\033[0m" for i in range(len(skater_names_grabbed))
    ]

    # Initialize PrettyTable with the dynamic headers
    fp_table = PrettyTable(fp_table_headers)
    fp_table.title = "\033[1mPlayer Fantasy Comparison\033[0m"

    table_fantasy_points_total = format_table_stats(skater_names_tot_fantasy_points, "Total Fantasy Points")
    fp_table.add_row(table_fantasy_points_total)

    table_fantasy_points_total = format_table_stats(skater_names_fantasy_points_per_game, "# of Fantasy Points per Game")
    fp_table.add_row(table_fantasy_points_total)

    print(fp_table)
        

    stats_headers = [
    f"\033[1m{skater_names_grabbed[i]}\033[0m" for i in range(len(skater_names_grabbed))
]

    # Initialize PrettyTable with the dynamic headers
    stats_table = PrettyTable(stats_headers)
    stats_table.title = "\033[1mPlayer Stats Comparison\033[0m"

    # games played
    table_season_games_played_total = format_table_stats(skater_season_gamesPlayed, "Games Played")
    stats_table.add_row(table_season_games_played_total)
    # average TOI
    table_season_toi_per_game = format_table_stats(skater_season_timeOnIcePerGame_min, "Average TOI")
    stats_table.add_row(table_season_toi_per_game)
    # goals
    stats_table.add_divider()
    table_season_goals_total = format_table_stats(skater_season_goals, "Total Goals")
    stats_table.add_row(table_season_goals_total)
    table_season_goals_per_game = format_table_stats(skater_season_goals_per_game, "# of Goals per Game")
    stats_table.add_row(table_season_goals_per_game)
    table_season_goals_per_60min = format_table_stats(skater_season_goals_per_60min, "# of Goals per 60 minutes")
    stats_table.add_row(table_season_goals_per_60min)
    # assists
    stats_table.add_divider()
    table_season_assists_total = format_table_stats(skater_season_assists, "Total Assists")
    stats_table.add_row(table_season_assists_total)
    table_season_assists_per_game = format_table_stats(skater_season_assists_per_game, "# of Assists per Game")
    stats_table.add_row(table_season_assists_per_game)
    table_season_assists_per_60min = format_table_stats(skater_season_assists_per_60min, "# of Assists per 60 minutes")
    stats_table.add_row(table_season_assists_per_60min)
    # +/-
    stats_table.add_divider()
    table_season_plusMinus_total = format_table_stats(skater_season_plusMinus, "Season +/-")
    stats_table.add_row(table_season_plusMinus_total)
    table_season_plusMinus_per_game = format_table_stats(skater_season_plusMinus_per_game, "Average +/- per Game")
    stats_table.add_row(table_season_plusMinus_per_game)
    table_season_plusMinus_per_60min = format_table_stats(skater_season_plusMinus_per_60min, "Average +/- per 60 minutes")
    stats_table.add_row(table_season_plusMinus_per_60min)
    # PP Goals
    stats_table.add_divider()
    table_season_ppGoals_total = format_table_stats(skater_season_ppGoals, "Season PP Goals")
    stats_table.add_row(table_season_ppGoals_total)
    table_season_ppGoals_per_game = format_table_stats(skater_season_ppGoals_per_game, "Average PP Goals per Game")
    stats_table.add_row(table_season_ppGoals_per_game)
    table_season_ppGoals_per_60min = format_table_stats(skater_season_ppGoals_per_60min, "Average # of PP Goals per 60 minutes")
    stats_table.add_row(table_season_ppGoals_per_60min)
    # PP Assists
    stats_table.add_divider()
    table_season_ppAssists_total = format_table_stats(skater_season_ppAssists, "Season PP Assists")
    stats_table.add_row(table_season_ppAssists_total)
    table_season_ppAssists_per_game = format_table_stats(skater_season_ppAssists_per_game, "Average PP Assists per Game")
    stats_table.add_row(table_season_ppAssists_per_game)
    table_season_ppAssists_per_60min = format_table_stats(skater_season_ppAssists_per_60min, "Average # of PP Assists per 60 minutes")
    stats_table.add_row(table_season_ppAssists_per_60min)
    # SH Goals
    stats_table.add_divider()
    table_season_shGoals_total = format_table_stats(skater_season_shGoals, "Season SH Goals")
    stats_table.add_row(table_season_shGoals_total)
    table_season_shGoals_per_game = format_table_stats(skater_season_shGoals_per_game, "Average SH Goals per Game")
    stats_table.add_row(table_season_shGoals_per_game)
    table_season_shGoals_per_60min = format_table_stats(skater_season_shGoals_per_60min, "# of SH Goals per 60 minutes")
    stats_table.add_row(table_season_shGoals_per_60min)
    # SH Assists
    stats_table.add_divider()
    table_season_shAssists_total = format_table_stats(skater_season_shAssists, "Season SH Assists")
    stats_table.add_row(table_season_shAssists_total)
    table_season_shAssists_per_game = format_table_stats(skater_season_shAssists_per_game, "Average SH Assists per Game")
    stats_table.add_row(table_season_shAssists_per_game)
    table_season_shAssists_per_60min = format_table_stats(skater_season_shAssists_per_60min, "Average # of SH Assists per 60 minutes")
    stats_table.add_row(table_season_shAssists_per_60min)
    # Game Winning Goals
    stats_table.add_divider()
    table_season_gameWinningGoals_total = format_table_stats(skater_season_gameWinningGoals, "Season Game Winning Goals")
    stats_table.add_row(table_season_gameWinningGoals_total)
    table_season_gameWinningGoals_per_game = format_table_stats(skater_season_gameWinningGoals_per_game, "Average # of Game Winning Goals per Game")
    stats_table.add_row(table_season_gameWinningGoals_per_game)
    table_season_gameWinningGoals_per_60min = format_table_stats(skater_season_gameWinningGoals_per_60min, "Average # of Game Winning Goals per 60 minutes")
    stats_table.add_row(table_season_gameWinningGoals_per_60min)
    # Shots
    stats_table.add_divider()
    table_season_shots_total = format_table_stats(skater_season_shots, "Season Shots")
    stats_table.add_row(table_season_shots_total)
    table_season_shots_per_game = format_table_stats(skater_season_shots_per_game, "Average # of Shots per Game")
    stats_table.add_row(table_season_shots_per_game)
    table_season_shots_per_60min = format_table_stats(skater_season_shots_per_60min, "Average # of Shots per 60 minutes")
    stats_table.add_row(table_season_shots_per_60min)
    # Hits
    stats_table.add_divider()
    table_season_hits_total = format_table_stats(skater_season_hits, "Season Hits")
    stats_table.add_row(table_season_hits_total)
    table_season_hits_per_game = format_table_stats(skater_season_hits_per_game, "Average # of Hits per Game")
    stats_table.add_row(table_season_hits_per_game)
    table_season_hits_per_60min = format_table_stats(skater_season_hits_per_60min, "Average # of Hits per 60 minutes")
    stats_table.add_row(table_season_hits_per_60min)
    # Blocked Shots
    stats_table.add_divider()
    table_season_blockedShots_total = format_table_stats(skater_season_blockedShots, "Season Blocked Shots")
    stats_table.add_row(table_season_blockedShots_total)
    table_season_blockedShots_per_game = format_table_stats(skater_season_blockedShots_per_game, "Average # of Blocked Shots per Game")
    stats_table.add_row(table_season_blockedShots_per_game)
    table_season_blockedShots_per_60min = format_table_stats(skater_season_blockedShots_per_60min, "Average # of Blocked Shots per 60 minutes")
    stats_table.add_row(table_season_blockedShots_per_60min)
    # FO Wins
    stats_table.add_divider()
    table_season_totalFaceoffWins_total = format_table_stats(skater_season_totalFaceoffWins, "Season FO Wins")
    stats_table.add_row(table_season_totalFaceoffWins_total)
    table_season_totalFaceoffWins_per_game = format_table_stats(skater_season_totalFaceoffWins_per_game, "Average # of FO Wins per Game")
    stats_table.add_row(table_season_totalFaceoffWins_per_game)
    table_season_totalFaceoffWins_per_60min = format_table_stats(skater_season_totalFaceoffWins_per_60min, "Average # of FO Wins per 60 minutes")
    stats_table.add_row(table_season_totalFaceoffWins_per_60min)
    # FO Loses
    stats_table.add_divider()
    table_season_totalFaceoffLosses_total = format_table_stats(skater_season_totalFaceoffLosses, "Season FO Losses")
    stats_table.add_row(table_season_totalFaceoffLosses_total)
    table_season_totalFaceoffLosses_per_game = format_table_stats(skater_season_totalFaceoffLosses_per_game, "Average # of FO Losses per Game")
    stats_table.add_row(table_season_totalFaceoffLosses_per_game)
    table_season_totalFaceoffLosses_per_60min = format_table_stats(skater_season_totalFaceoffLosses_per_60min, "Average # of FO Losses per 60 minutes")
    stats_table.add_row(table_season_totalFaceoffLosses_per_60min)
    # PIMs
    stats_table.add_divider()
    table_season_penaltyMinutes_total = format_table_stats(skater_season_penaltyMinutes, "Season PIMs")
    stats_table.add_row(table_season_penaltyMinutes_total)
    table_season_penaltyMinutes_per_game = format_table_stats(skater_season_penaltyMinutes_per_game, "Average # of PIMs per Game")
    stats_table.add_row(table_season_penaltyMinutes_per_game)
    table_season_penaltyMinutes_per_60min = format_table_stats(skater_season_penaltyMinutes_per_60min, "Average # of PIMs per 60 minutes")
    stats_table.add_row(table_season_penaltyMinutes_per_60min)

    print(stats_table)

In [6]:
comp_multi_skaters(skater_names,
    season,
    fp_goals,
    fp_assists,
    fp_plusminus,
    fp_pp_goals,
    fp_pp_assists,
    fp_sh_goals,
    fp_sh_assists,
    fp_game_winning_goals,
    fp_shots,
    fp_hits,
    fp_blocks,
    fp_fowins,
    fp_folosses,
    fp_pims,
    )

+----------------------------------------------------------------------------------------+
|                               Player Fantasy Comparison                                |
+------------------------------+-------------+---------------+------------+--------------+
|             Stat             | Collin Graf | Tyler Toffoli | Will Smith | Michael Misa |
+------------------------------+-------------+---------------+------------+--------------+
|     Total Fantasy Points     |    432.2    |     433.45    |   467.7    |    181.35    |
| # of Fantasy Points per Game |     5.34    |      5.49     |    6.78    |     4.03     |
+------------------------------+-------------+---------------+------------+--------------+
+----------------------------------------------------------------------------------------------------------+
|                                         Player Stats Comparison                                          |
+------------------------------------------------+----

In [ ]:
skater_names_tot_fantasy_points_obj = skater_names_tot_fantasy_points.astype(object)
skater_names_tot_fantasy_points_obj = np.insert(skater_names_tot_fantasy_points_obj, 0, "Total Fantasy Points")
skater_names_tot_fantasy_points = np.insert(skater_names_tot_fantasy_points, 0, -9999.)

skater_names_fantasy_points_per_game_obj = skater_names_fantasy_points_per_game.astype(object)
skater_names_fantasy_points_per_game_obj = np.insert(skater_names_fantasy_points_per_game_obj, 0, "Fantasy Points / Game")
skater_names_fantasy_points_per_game = np.insert(skater_names_fantasy_points_per_game, 0, -9999.)


In [ ]:
skater_names_grabbed

In [ ]:
skater_names_tot_fantasy_points_obj

In [ ]:
skater_names_fantasy_points_per_game

In [ ]:
# Create dynamic column names based on array length, bolding the largest index
"""
headers = [
    f"\033[1m{skater_names_grabbed[i]}\033[0m" if i != idx_max_fp_per_game else f"\033[1m{skater_names_grabbed[i]}\033[0m"
    for i in range(len(skater_names_grabbed))
]
"""
headers = [
    f"\033[1m{skater_names_grabbed[i]}\033[0m" for i in range(len(skater_names_grabbed))
]

# Initialize PrettyTable with the dynamic headers
table = PrettyTable(headers)
table.title = "\033[1mPlayer Fantasy Comparison\033[0m"


# Find index of the largest value
idx_max_fp = np.argmax(skater_names_tot_fantasy_points)
# Create row data, bolding the largest value
row_data = [
    str(val) if i != idx_max_fp else f"\033[1m{val}\033[0m"
    for i, val in enumerate(skater_names_tot_fantasy_points_obj)
]
# Add the row and display
table.add_row(row_data)

# Find index of the largest value
idx_max_fp_per_game = np.argmax(skater_names_fantasy_points_per_game)
# Create row data, bolding the largest value
row_data = [
    str(val) if i != idx_max_fp_per_game else f"\033[1m{val}\033[0m"
    for i, val in enumerate(skater_names_fantasy_points_per_game_obj)
]
# Add the row and display
table.add_row(row_data)

print(table)


In [ ]:
# Find index of the largest value
idx_max_fp_per_game = np.argmax(skater_names_fantasy_points_per_game)
print(idx_max_fp_per_game)

# Create dynamic column names based on array length, bolding the largest index
headers = [
    f"{skater_names_grabbed[i]}" if i != idx_max_fp_per_game else f"\033[1m{skater_names_grabbed[i]}\033[0m"
    for i in range(len(skater_names_grabbed))
]

# Initialize PrettyTable with the dynamic headers
table = PrettyTable(headers)
table.title = "Fantasy Points Per Game"

# Create row data, bolding the largest value
row_data = [
    str(val) if i != idx_max_fp_per_game else f"\033[1m{val}\033[0m"
    for i, val in enumerate(skater_names_fantasy_points_per_game)
]

# Add the row and display
table.add_row(row_data)

print(table)


In [ ]:
from prettytable import PrettyTable

# Define your input array
my_array = [14, 25, 89, 42, 19]

# Find index of the largest value
max_index = my_array.index(max(my_array))

# Create dynamic column names based on array length, bolding the largest index
headers = [
    f"Index {i}" if i != max_index else f"\033[1mIndex {i}\033[0m"
    for i in range(len(my_array))
]

# Initialize PrettyTable with the dynamic headers
table = PrettyTable(headers)

# Create row data, bolding the largest value
row_data = [
    str(val) if i != max_index else f"\033[1m{val}\033[0m"
    for i, val in enumerate(my_array)
]

# Add the row and display
table.add_row(row_data)
print(table)


In [ ]:
skater_names_grabbed

In [ ]:
skater_season_gamesPlayed

In [ ]:
skater_season_totalFaceoffWins

In [ ]:
skater_summary_query[0]

In [ ]:
import numpy as np
import pandas as pd

# 1. Define your input array
my_array = [10, 45, 22, 89, 34]

# 2. Determine number of columns based on array length
num_cols = len(my_array)

# 3. Find the index of the largest value in the array
max_index = int(np.argmax(my_array))

# 4. Create dynamic column names and a single-row dataframe
col_names = [f'Col_{i}' for i in range(num_cols)]
df = pd.DataFrame([my_array], columns=col_names)


# 5. Define a styling function to bold the specific column
def bold_max_col(val, col_idx):
  if col_idx == max_index:
    return 'font-weight: bold; background-color: #f0f0f0;'
  return ''


# Apply style and display in Jupyter
styled_df = df.style.apply(
    lambda col: [
        bold_max_col(v, df.columns.get_loc(col.name)) for v in col
    ],
    axis=0,
)
styled_df
